# Notebook 0 — Quantum Bits from Scratch

*Part of **QEC Explorer**, and the gentlest starting point. Before we build error-correcting codes, let's build the thing they protect: a single **qubit**. In about 30 lines of NumPy you'll represent a qubit, flip it the two ways noise can, measure it (and watch measurement destroy information), and see why you can't copy it. No prior quantum knowledge needed.*

This is the code companion to the interactive **[Quantum Basics](basics.html)** page. Same four ideas, but here you can see the actual math.

---
## 1 · A qubit is just two numbers

An ordinary bit is `0` or `1`. A **qubit** is described by **two numbers** (called *amplitudes*): one for "how much 0" and one for "how much 1." We write it as a 2-element vector:

$$|\psi\rangle = \begin{pmatrix} a_0 \\ a_1 \end{pmatrix}, \qquad |a_0|^2 + |a_1|^2 = 1.$$

The squared sizes are **probabilities**: $|a_0|^2$ is the chance you'd measure 0, $|a_1|^2$ the chance of 1. So they must add to 1.

In [1]:
import numpy as np
np.random.seed(0)

# The two "definite" states:
ket0 = np.array([1.0, 0.0])   # |0> : measure 0 with probability 1
ket1 = np.array([0.0, 1.0])   # |1> : measure 1 with probability 1

# An even blend (the famous "superposition"): equal amplitude on 0 and 1.
plus = np.array([1.0, 1.0]) / np.sqrt(2)   # |+> : 50/50

def probs(state):
    # Born rule: probability of each outcome is the squared amplitude.
    return np.abs(state) ** 2

for name, s in [("|0>", ket0), ("|1>", ket1), ("|+>", plus)]:
    p = probs(s)
    print(f"{name}: amplitudes {np.round(s,3)}  ->  P(0)={p[0]:.2f}, P(1)={p[1]:.2f}")

|0>: amplitudes [1. 0.]  ->  P(0)=1.00, P(1)=0.00
|1>: amplitudes [0. 1.]  ->  P(0)=0.00, P(1)=1.00
|+>: amplitudes [0.707 0.707]  ->  P(0)=0.50, P(1)=0.50


---
## 2 · Errors are matrices: the X flip and the Z flip

Noise acts on a qubit by **multiplying its vector by a matrix**. The two most basic errors, the ones the whole rest of QEC Explorer is built on, are:

- **X (bit-flip):** swaps the 0 and 1 amplitudes. $X|0\rangle = |1\rangle$.
- **Z (phase-flip):** leaves 0 alone but flips the *sign* of the 1 amplitude. Invisible to a plain 0/1 measurement, but real.

In [2]:
X = np.array([[0.0, 1.0],
              [1.0, 0.0]])   # bit-flip: swaps the two amplitudes
Z = np.array([[1.0,  0.0],
              [0.0, -1.0]])  # phase-flip: negates the |1> amplitude

print("X|0> =", X @ ket0, " (that's |1> — the bit flipped)")
print("X|1> =", X @ ket1, " (back to |0>)")
print("Z|0> =", Z @ ket0, " (unchanged)")
print("Z|+> =", np.round(Z @ plus, 3), " (the |1> part went negative — a phase flip)")

# A Z flip does NOT change the measurement probabilities of |+>:
print("\nP of |+>     :", np.round(probs(plus), 3))
print("P of Z|+>    :", np.round(probs(Z @ plus), 3), " <- identical! Z is invisible to a 0/1 measurement.")

X|0> = [0. 1.]  (that's |1> — the bit flipped)
X|1> = [1. 0.]  (back to |0>)
Z|0> = [1. 0.]  (unchanged)
Z|+> = [ 0.707 -0.707]  (the |1> part went negative — a phase flip)

P of |+>     : [0.5 0.5]
P of Z|+>    : [0.5 0.5]  <- identical! Z is invisible to a 0/1 measurement.


That last line is the key subtlety: a **Z error on |+⟩ changes the state but not the 0/1 probabilities.** A naive "just measure it" check would miss it entirely. Real codes are built to catch both X *and* Z.

---
## 3 · Measuring a qubit (and why it destroys information)

You can't read the amplitudes directly. **Measurement** returns a single bit, 0 or 1, chosen at random with probability $|a_0|^2$ / $|a_1|^2$, and the qubit **collapses** to that outcome. The blend is gone.

Let's measure `|+>` many times and watch it land ~50/50, then confirm the collapse.

In [3]:
def measure(state, rng=np.random):
    # sample an outcome by the Born rule, then collapse the state to it
    p = probs(state)
    outcome = rng.choice([0, 1], p=p)
    collapsed = ket0 if outcome == 0 else ket1
    return outcome, collapsed

rng = np.random.default_rng(1)
N = 2000
counts = {0: 0, 1: 0}
for _ in range(N):
    out, _ = measure(plus, rng)
    counts[out] += 1
print(f"Measured |+> {N} times: {counts[0]} zeros, {counts[1]} ones "
      f"({100*counts[0]/N:.1f}% / {100*counts[1]/N:.1f}%) -- about 50/50, as predicted.")

# The collapse: once measured, it's stuck.
out, after = measure(plus, rng)
print(f"\nOne measurement gave {out}; the state is now {after} (definitely {out}).")
print("The original 50/50 blend is destroyed. You cannot un-measure it.")

Measured |+> 2000 times: 1002 zeros, 998 ones (50.1% / 49.9%) -- about 50/50, as predicted.

One measurement gave 0; the state is now [1. 0.] (definitely 0).
The original 50/50 blend is destroyed. You cannot un-measure it.


**This is the central obstacle of quantum error correction.** With ordinary bits you'd just read your data to check for errors. Here, reading the data destroys it. The surface code's whole trick (Module 1) is checking for errors *without ever measuring the protected information*.

---
## 4 · Why you can't keep a backup: no-cloning

The obvious defense against errors is redundancy: keep three copies and majority-vote. For qubits this is **impossible**, the *no-cloning theorem*. Here's the honest one-line argument, which you can verify: a single operation (a single matrix) cannot copy *every* qubit.

Suppose a "cloner" matrix `C` existed that copied any state into a two-qubit system. It would have to satisfy `C|0> -> |0>|0>` and `C|1> -> |1>|1>`. But then, because matrices are **linear**, it would be forced to send `|+> = (|0>+|1>)/√2` to `(|0>|0> + |1>|1>)/√2`, which is *not* two copies of `|+>` (two copies would be `|+>|+>`). The two disagree, so no such `C` works for all states.

In [4]:
# Demonstrate the contradiction numerically with 2-qubit (4-dim) vectors.
# Basis order: |00>, |01>, |10>, |11>.
def kron(a, b):
    return np.kron(a, b)

# What a cloner is REQUIRED to do on the basis states:
clone_of_0 = kron(ket0, ket0)   # |0> -> |00>
clone_of_1 = kron(ket1, ket1)   # |1> -> |11>

# Linearity then FORCES this for |+> = (|0>+|1>)/sqrt2:
forced = (clone_of_0 + clone_of_1) / np.sqrt(2)     # = (|00>+|11>)/sqrt2

# But a real copy of |+> would be |+>|+>:
real_copy = kron(plus, plus)

print("Linearity forces C|+> =", np.round(forced, 3))
print("A true copy   |+>|+> =", np.round(real_copy, 3))
print("\nThese are different vectors, so no single 'cloner' matrix can copy every qubit.")
print("Are they equal?", np.allclose(forced, real_copy), " <- False = no-cloning, proven.")

Linearity forces C|+> = [0.707 0.    0.    0.707]
A true copy   |+>|+> = [0.5 0.5 0.5 0.5]

These are different vectors, so no single 'cloner' matrix can copy every qubit.
Are they equal? False  <- False = no-cloning, proven.


So there are **no quantum backups**. Error correction can't copy a qubit; instead it **spreads one logical qubit across many physical qubits** in an entangled pattern, so a few errors can be detected and reversed. That pattern is the surface code, and that's exactly where Notebook 1 picks up.

---
## 5 · Proof: this matches the basics

A quick self-check that everything above is internally consistent and matches the conventions used across the rest of QEC Explorer.

In [5]:
def check(name, cond):
    print(f"  {'OK ' if cond else 'XX '} {name}")
    assert cond, name

print("Self-checks:\n")
check("|0>, |1>, |+> are all normalized (probabilities sum to 1)",
      all(abs(probs(s).sum() - 1) < 1e-12 for s in [ket0, ket1, plus]))
check("X swaps |0> and |1>", np.allclose(X @ ket0, ket1) and np.allclose(X @ ket1, ket0))
check("X is its own inverse (two flips = identity)", np.allclose(X @ X, np.eye(2)))
check("Z leaves |0> alone, negates |1>", np.allclose(Z @ ket0, ket0) and np.allclose(Z @ ket1, -ket1))
check("Z is invisible to a 0/1 measurement of |+>", np.allclose(probs(plus), probs(Z @ plus)))
check("X and Z anticommute (XZ = -ZX) -- the root of why two error types exist",
      np.allclose(X @ Z, -(Z @ X)))
check("no-cloning: forced C|+> != a true copy |+>|+>",
      not np.allclose((np.kron(ket0,ket0)+np.kron(ket1,ket1))/np.sqrt(2), np.kron(plus,plus)))
print("\nAll checks pass. You've built a qubit, its two errors, measurement, and no-cloning from scratch.")

Self-checks:

  OK  |0>, |1>, |+> are all normalized (probabilities sum to 1)
  OK  X swaps |0> and |1>
  OK  X is its own inverse (two flips = identity)
  OK  Z leaves |0> alone, negates |1>
  OK  Z is invisible to a 0/1 measurement of |+>
  OK  X and Z anticommute (XZ = -ZX) -- the root of why two error types exist
  OK  no-cloning: forced C|+> != a true copy |+>|+>

All checks pass. You've built a qubit, its two errors, measurement, and no-cloning from scratch.


---
## 6 · Wrap-up — on to detection

You now have the whole vocabulary the rest of the project uses:

- a **qubit** is a 2-vector of amplitudes; squared amplitudes are measurement probabilities,
- the two basic **errors** are **X** (bit-flip) and **Z** (phase-flip), and they're just matrices,
- **measuring** collapses the state, so you can't simply read your data to check it,
- and **no-cloning** means you can't keep a backup, so codes spread information out instead.

That's exactly enough to start watching a real code at work:

- **→ Notebook 1 — Building the surface code by hand** turns these single qubits into a full error-correcting code.
- **→ [Quantum Basics (interactive)](basics.html)** is the click-through version of this notebook.
- **→ [Module 1 · Detection (live)](https://github.com/kondshk/QEC-Explorer)** lets you inject the X and Z errors you just met and watch them get caught.

Welcome to quantum error correction.